<a href="https://colab.research.google.com/github/rileyrdunn17/priority-atx-food-safety-app/blob/main/priority_atx_food_safety_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Priority ATX Food Safety App
My app helps Austin restaurant goers decide which restaurants are the cleanest and worth visiting by evaluating existing restaurant inspection grades and health ratings.

In [21]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import os
import time
import pandas as pd

# Look for food_establishment_inspection_scores.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "food_establishment_inspection_scores.csv")

if len(found) == 0:
    print("❌ Can't find food_establishment_inspection_scores.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    scores = pd.read_csv(FOLDER + "food_establishment_inspection_scores.csv")
    print(f"✅ Loaded {len(scores)} scores from:", FOLDER)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded 20421 scores from: /content/drive/MyDrive/Year 3/MIST 4600/MIST 4600 Project/


In [22]:
def top_rated(scores, n):
    best = scores.sort_values("Score", ascending=False)
    return best[["Restaurant Name", "Score"]].head(n)

print(top_rated(scores, 5))

                       Restaurant Name  Score
20365             OOB - BC - McDonalds  100.0
46     Sneha's Vegetarian Kitchen, LLC  100.0
20419          Lake Travis High School  100.0
0                            Drury Inn  100.0
12                    Taco Bell #30150  100.0


In [23]:

def scores_by_zip(scores):
  scores["first_5_zip_code"] = scores["Zip Code"].str[:5]
  return scores["first_5_zip_code"].value_counts()

print(scores_by_zip(scores))

first_5_zip_code
78660    1477
78701    1469
78753    1154
78758     973
78745     824
78704     809
78723     764
78702     749
78705     716
78738     649
78759     640
78757     635
78741     592
78752     576
78744     557
78748     542
78751     517
78746     517
78703     488
78653     439
78734     391
78727     380
78729     371
78749     371
78756     316
78754     308
78731     285
78735     261
78750     256
78724     252
78717     246
78617     235
78728     193
78719     172
78726     160
78721     146
78732     143
78722     128
78613      86
78736      83
78669      83
78725      60
78747      55
78739      52
78733      52
78737      45
78652      41
78664      34
78730      33
78620      18
78634      18
78742      14
78610       9
78641       6
78621       6
78615       5
78654       4
78612       4
78645       1
Name: count, dtype: int64


In [24]:
scores["first_5_zip_code"] = scores["Zip Code"].str[:5]
def avg_score(scores):
  answer = scores.groupby ("first_5_zip_code")["Score"].mean()
  return answer

print(avg_score(scores))


first_5_zip_code
78610    97.666667
78612    83.250000
78613    89.722892
78615    83.800000
78617    92.800000
78620    94.722222
78621    89.833333
78634    93.333333
78641    85.166667
78645    88.000000
78652    94.317073
78653    90.742009
78654    87.000000
78660    92.828008
78664    93.000000
78669    93.650602
78701    88.923393
78702    89.582329
78703    90.608607
78704    90.004944
78705    90.033566
78717    90.592593
78719    91.205882
78721    92.527397
78722    89.937500
78723    90.387435
78724    90.852590
78725    97.355932
78726    94.187500
78727    91.111111
78728    92.295337
78729    92.549865
78730    91.909091
78731    91.612676
78732    94.272727
78733    93.980769
78734    90.497436
78735    91.808429
78736    92.301205
78737    92.444444
78738    92.949074
78739    97.750000
78741    89.768581
78742    96.928571
78744    91.971275
78745    89.522644
78746    93.170543
78747    90.581818
78748    91.704797
78749    93.684636
78750    91.027344
78751    91.08

In [29]:
import requests

from google.colab import userdata
key = userdata.get("CENSUS_API_KEY")

year = 2024
url = "https://api.census.gov/data/2024/acs/acs5"
params = {"get":"NAME,B19013_001E", "for":"zip code tabulation area:*", "key":key}

rows = requests.get(url, params=params).json()
census = pd.DataFrame(rows[1:],columns=rows[0])
census = census.rename(columns={"B19013_001E":"income", "zip code tabulation area": "zip"})
census["income"] = pd.to_numeric(census["income"])
census = census[census["income"]>0]


insp = pd.read_csv(FOLDER + "food_establishment_inspection_scores.csv")
insp["zip"]=insp["Zip Code"].astype(str).str[:5]
df = insp.merge(census, on="zip")

zipdf= df.groupby("zip").agg(income=("income","first"), average_score=("Score","mean"),inspections=("Score", "size")).reset_index()

print(zipdf.sort_values("income"))

print(df[["income","Score"]].corr())
print(zipdf[["income","average_score"]].corr())


      zip  income  average_score  inspections
20  78705   32380      90.033566          716
22  78719   56202      91.205882          172
52  78753   65170      90.150043         1154
42  78741   66741      89.768581          592
51  78752   67682      89.744348          576
4   78617   71640      92.800000          235
12  78654   74130      87.000000            4
56  78758   74381      91.964948          973
30  78728   75243      92.295337          193
26  78724   77926      90.852590          252
50  78751   80915      91.085106          517
3   78615   80938      83.800000            5
43  78744   83002      91.971275          557
23  78721   84573      92.527397          146
44  78745   86109      89.522644          824
31  78729   86438      92.549865          371
14  78664   87417      93.000000           34
53  78754   90121      92.142857          308
28  78726   90885      94.187500          160
46  78747   92748      90.581818           55
38  78736   93805      92.301205  